# T-circuit discovery: patch every head on the note contrast (Qwen2.5-VL-3B)

Owner: **@SohamRShinde**. Design pre-registered in `docs/DECISIONS.md` **D71**
before this notebook ran; the reading of the result is fixed by **D70**.

This is `replication_patching.ipynb`'s pipeline -- `LastTokenRunner`,
`patch_example`, `classify_heads`, all **unchanged** (D30, D68) -- with a
different pair of inputs, so the P and T circuits come out of one pipeline:

| | source (competing signal wins) | target (image wins) |
|---|---|---|
| P (Kush, done) | Prior prompt, counterfactual image | Visual prompt, counterfactual image |
| **T (this)** | Visual prompt, **incongruent note** | Visual prompt, **neutral note** |

Both T renders use the `on_object_small` layout on the original image and
differ **only in the note's word**. `patch_example`'s record fields keep their
"prior"/"visual" names, so read them as **prior = incongruent, visual =
neutral** (`t_patching.T_ROLES`).

**Phases**, each checkpointed one JSON line per object (D44):

1. **Select** (~45 min): both renders of all 469 objects on the cached path;
   keep objects right on the neutral note and fooled by the incongruent one.
2. **Sweep** (~7 h): patch all 576 heads both ways on each selected object,
   objects shared with P first, then the rest in seeded random order (D71) --
   so a cut-off run is still usable.
3. **Classify and save.**

**Runtime ~8-9 h. Run it as Save Version -> Save & Run All (Commit).**
Section 2 times one patched step and prints the real estimate.

**To resume after a cut-off:** attach the previous version's output as an
input dataset and re-run everything; section 2 copies its `t_patching/*.jsonl`
checkpoints back and the loops skip finished objects.

Output prefix: `/kaggle/working/t_patching/`. Settings: GPU T4, Internet ON,
Kaggle secret `HF_TOKEN`.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
import importlib.util as _ilu
_need = {}
for _mod, _names in {
    "circuit_overlap.t_patching": ["render_t_pair", "is_t_conflicting", "sweep_order",
                                   "finite_records", "screen_cross_check", "LAYOUT", "T_ROLES"],
    "circuit_overlap.patching": ["LastTokenRunner", "patch_example", "mean_restoration_grids",
                                 "classify_heads", "capture_last_token_o_proj_inputs",
                                 "patch_head_last_token"],
    "circuit_overlap.replication": ["lietzow_rows", "run_cached"],
}.items():
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = ["module"]
        continue
    _m = importlib.import_module(_mod)
    _gone = [n for n in _names if not hasattr(_m, n)]
    if _gone:
        _need[_mod] = _gone
if _need:
    raise RuntimeError(f"checkout {COMMIT} is missing {_need}. "
                       "Push the local commits to origin/main, then re-run this cell.")
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). Installing it into a
# kernel that already imported transformers does nothing: restart and re-run.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
from circuit_overlap.t_screen import native_bf16
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
# Not torch.cuda.is_bf16_supported(): it says True on a T4 (emulation).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
from circuit_overlap.t_screen import ScreenConfig, VISUAL_PROMPT, load_model
config = ScreenConfig()
model, processor = load_model(config)        # 4-bit nf4, double quant, fp16 compute on a T4 (D10)
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

text_config = model.model.language_model.config
n_layers = text_config.num_hidden_layers
n_heads = text_config.num_attention_heads
head_dim = getattr(text_config, "head_dim", text_config.hidden_size // n_heads)
print(f"{n_layers} layers x {n_heads} heads, head_dim={head_dim}")
assert (n_layers, n_heads) == (36, 16), "not the geometry P was discovered on"

## 1b. [VERIFY] Image size and speed

Same check as `replication_patching.ipynb`: the image token budget must be
~1,280 (pinned area, as Lietzow), and a forward pass must take seconds. The
input builder is identical to P's, so the two conditions are tokenised alike.

In [ ]:
import time, torch
from datasets import load_dataset
from transformers import AutoProcessor

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_probe = ds[0]["original_image"].convert("RGB")

def build_inputs(image, prompt):
    messages = [{"role": "user",
                 "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[image], return_tensors="pt").to(model.device)

def n_image_tokens(image):
    t, h, w = build_inputs(image, "x")["image_grid_thw"][0].tolist()
    return t * h * w // 4                    # 2x2 patch merge -> LLM tokens

EXPECTED = config.pixels // (28 * 28)        # 1280
n_img = n_image_tokens(_probe)
print(f"image {_probe.size} -> {n_img} image tokens (expected ~{EXPECTED})")
if n_img > 1.2 * EXPECTED:
    processor = AutoProcessor.from_pretrained(
        config.checkpoint, use_fast=True,
        size={"shortest_edge": config.pixels, "longest_edge": config.pixels},
    )
    n_img = n_image_tokens(_probe)
    print(f"rebuilt the processor with size=... -> {n_img} image tokens")
assert n_img <= 1.2 * EXPECTED, "image token budget is still wrong -- do not run the loops"

assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 and restart"
_inp = build_inputs(_probe, VISUAL_PROMPT.format(object="banana"))
for attempt in ("warm-up", "timed"):
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        model(**_inp, use_cache=False)
    torch.cuda.synchronize()
    print(f"{attempt} forward pass: {time.time() - t0:.1f}s")

## 2. Objects, one sample pair, and a check of the cached path

Same row filters and colour tokens as P (`lietzow_rows`,
`color_first_token_ids`). Look at the sample pair: the two images must differ
only in the note's word. The cached last-token path must equal a full forward
pass on **both** renders, plain and with one head patched; the cell stops the
notebook if not.

In [ ]:
import glob, inflect, json, pathlib, shutil
from collections import Counter
from IPython.display import display
from circuit_overlap.replication import lietzow_rows, run_cached
from circuit_overlap.t_screen import color_first_token_ids, load_notes_config
from circuit_overlap.patching import (
    LastTokenRunner, capture_last_token_o_proj_inputs, patch_head_last_token, patch_example,
)
from circuit_overlap.t_patching import LAYOUT, T_ROLES, render_t_pair, is_t_conflicting

WORK = pathlib.Path("/kaggle/working/t_patching"); WORK.mkdir(parents=True, exist_ok=True)

# Resume: bring back checkpoints from an attached previous run's output.
for name in ("selection.jsonl", "records.jsonl"):
    found = sorted(glob.glob(f"/kaggle/input/**/t_patching/{name}", recursive=True))
    if found and not (WORK / name).exists():
        shutil.copy(found[0], WORK / name)
        print(f"resuming from {found[0]}")

_eng = inflect.engine()
rows, dropped = lietzow_rows(ds, _eng)
row_by_i = {r["i"]: r for r in rows}
print(len(rows), "objects", dict(dropped), "(P used the same 469)")
vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
ambiguous = set(collisions)
print(len(vocab), "colours | sharing a first token (D26):", collisions)

notes_cfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")
spec = next(s for s in notes_cfg["specs"] if s.name == LAYOUT)
neutral_words = notes_cfg["neutral_words"]
print("layout:", spec, "\nroles:", T_ROLES)

P_FILE = f"{REPO_DIR}/results/replication_patching_qwen25vl3b.json"
p_ids = [r["i"] for r in json.load(open(P_FILE))["result"]["records"]]
print(len(p_ids), "P objects (Kush's run at 90286fa)")

def pair_inputs(r):
    """(incongruent inputs, neutral inputs, render info) for one object."""
    row = ds[r["i"]]
    pr = render_t_pair(row["original_image"], row["counterfact_image"],
                       r["attack"], neutral_words, spec)
    prompt = VISUAL_PROMPT.format(object=r["name"])
    return build_inputs(pr["incongruent"], prompt), build_inputs(pr["neutral"], prompt), pr

# [VERIFY] one sample pair, by eye: only the word on the note may differ.
r0 = rows[0]
src0, tgt0, pr0 = pair_inputs(r0)
print(f"{r0['name']}: correct {r0['correct']}, note says {r0['attack']!r} vs neutral "
      f"{pr0['neutral_word']!r} | occlusion {pr0['object_occlusion']:.1%} | "
      f"same footprint {pr0['same_note_footprint']}")
display(pr0["neutral"].resize((320, 320))); display(pr0["incongruent"].resize((320, 320)))
assert pr0["same_note_footprint"], "the two notes differ in shape: not a single-factor contrast"
assert src0["input_ids"].shape == tgt0["input_ids"].shape, "renders tokenise to different lengths"

# [VERIFY] cached path == full forward pass, on both renders, plain and patched.
source0, target0 = LastTokenRunner(model, src0), LastTokenRunner(model, tgt0)
print("incongruent plain   max |dlogit|:", source0.check_against_full_forward(atol=1.0))
print("neutral     plain   max |dlogit|:", target0.check_against_full_forward(atol=1.0))
with capture_last_token_o_proj_inputs(model) as acts0:
    source0.logits()
layer, head = 26, 5
value = acts0[layer][head * head_dim:(head + 1) * head_dim]
with patch_head_last_token(model, layer, head, head_dim, value):
    print("neutral     patched max |dlogit|:", target0.check_against_full_forward(atol=1.0))

# Time one patched step and estimate the whole run.
t0 = time.time()
for _ in range(10):
    with patch_head_last_token(model, layer, head, head_dim, value):
        target0.logits()
torch.cuda.synchronize()
step = (time.time() - t0) / 10
per_object = 2 * n_layers * n_heads * step
print(f"one patched step: {step * 1000:.0f} ms -> {per_object:.0f} s per object")
for n_sel in (200, 250, 300):
    print(f"  if {n_sel} objects are selected: ~{n_sel * per_object / 3600:.1f} h of patching")
print("plus ~45 min to select and ~30 min of prompt runs in the sweep")
del source0, target0, acts0

## 3. Phase A: select objects (all 469, about 45 min)

T's "correctly conflicting" (D71): the neutral run's top-1 is a real colour and
the incongruent run's top-1 is the note's colour, as first-token ids, on the
**cached path** -- the same path the sweep patches, so a near-tie cannot select
an object here and then behave differently there.

In [ ]:
def select_object(r):
    src_in, tgt_in, pr = pair_inputs(r)
    out = {"name": r["name"], "attack": r["attack"], "neutral_word": pr["neutral_word"],
           "object_occlusion": pr["object_occlusion"], "note_area_frac": pr["note_area_frac"]}
    if src_in["input_ids"].shape != tgt_in["input_ids"].shape:
        return {"status": "shape_mismatch", **out}
    source, target = LastTokenRunner(model, src_in), LastTokenRunner(model, tgt_in)
    logits_inc, logits_neu = source.logits(), target.logits()
    if not (torch.isfinite(logits_inc).all() and torch.isfinite(logits_neu).all()):
        return {"status": "numerics_failed", **out}       # plan risk 7
    top1_inc, top1_neu = int(logits_inc.argmax()), int(logits_neu.argmax())
    out.update(top1_incongruent=top1_inc, top1_neutral=top1_neu)
    ok = is_t_conflicting(r, top1_neu, top1_inc, color_ids, ambiguous)
    return {"status": "selected" if ok else "not_conflicting", **out}

selection = run_cached(WORK / "selection.jsonl", rows, select_object, progress_every=50)
status = Counter(s["status"] for s in selection)
print(status)
selected = [s["i"] for s in selection if s["status"] == "selected"]
print(f"{len(selected)} selected of {len(rows)}")

# Sanity check against the behavioural screen (different path and dtype, so
# agreement is expected to be high but not perfect; not a criterion).
from circuit_overlap.t_patching import screen_cross_check
screen_ids = json.load(open(f"{REPO_DIR}/results/t_usable_objects.json"))["result"]["usable_object_ids"]
print("cross-check with the screen:", screen_cross_check(selected, screen_ids))
print("selected objects also in P:", len(set(selected) & set(p_ids)), "of", len(p_ids))

## 4. Phase B: patch every head (about 7 h)

Objects shared with P first, then the rest in seeded random order (D71). Each
object re-renders its pair and confirms its top-1 answers still match the
selection, then `patch_example` runs both directions over all 576 heads with
**source = incongruent, target = neutral**. Records are rounded to 4 decimals,
as P's are.

In [ ]:
from circuit_overlap.t_patching import sweep_order, has_nonfinite

order = sweep_order(selected, p_ids, seed=0)
todo = [row_by_i[i] for i in order]
selected_top1 = {s["i"]: (s["top1_incongruent"], s["top1_neutral"])
                 for s in selection if s["status"] == "selected"}
print(f"{len(todo)} objects to sweep; the first {len(set(selected) & set(p_ids))} are shared with P")

def sweep_object(r):
    src_in, tgt_in, _ = pair_inputs(r)
    source, target = LastTokenRunner(model, src_in), LastTokenRunner(model, tgt_in)
    top1 = (int(source.logits().argmax()), int(target.logits().argmax()))
    out = {"name": r["name"], "top1_incongruent": top1[0], "top1_neutral": top1[1]}
    if top1 != selected_top1[r["i"]]:
        return {"status": "selection_drift", **out}       # should never happen
    # Roles (T_ROLES): source = incongruent note, target = neutral note.
    record = patch_example(model, source, target, head_dim)
    del source, target
    if record is None:                                     # argmax tokens coincide
        return {"status": "same_argmax", **out}
    for key, value in record.items():
        if isinstance(value, list):
            record[key] = [[round(x, 4) for x in row] for row in value]
    full = {"status": "patched", **out, **record}
    if has_nonfinite(full):
        full["status"] = "numerics_failed"                 # plan risk 7
    return full

sweep = run_cached(WORK / "records.jsonl", todo, sweep_object, progress_every=5)
print(Counter(s["status"] for s in sweep))

## 5. Classify the T heads and save

`classify_heads` is the same PCA +/- 2 sigma rule that found P's heads. The
J_PT printed here is **preliminary**: the reported numbers, with bootstrap
intervals and D70's wording, come from the analysis script run on the files
saved below.

In [ ]:
import gzip
import numpy as np
from circuit_overlap.patching import mean_restoration_grids, classify_heads, rank_heads
from circuit_overlap.t_patching import finite_records
from circuit_overlap.metrics import jaccard, containment
from circuit_overlap.heads import format_head_label
from circuit_overlap.provenance import result_envelope, write_result

patched = [s for s in sweep if s["status"] == "patched"]
patched, nonfinite = finite_records(patched)
p2v, v2p, n_used = mean_restoration_grids(patched)
cls = classify_heads(p2v, v2p, sigma=2.0)
t_prom, t_supp = set(cls["promoting"]), set(cls["suppressing"])

p_res = json.load(open(P_FILE))["result"]
p_prom = {tuple(h) for h in p_res["promoting"]}
print(f"{n_used} objects scored ({len(nonfinite)} dropped as non-finite)")
print("T promoting  :", sorted(format_head_label(h) for h in t_prom))
print("T suppressing:", sorted(format_head_label(h) for h in t_supp))
print("P promoting  :", sorted(format_head_label(h) for h in p_prom))
print(f"PRELIMINARY  J_PT {jaccard(p_prom, t_prom):.3f} | containment P in T "
      f"{containment(p_prom, t_prom):.2f} | read against D70 via the analysis script")

summary = {
    "n_rows": len(rows), "dropped": dict(dropped),
    "selection_status": dict(Counter(s["status"] for s in selection)),
    "sweep_status": dict(Counter(s["status"] for s in sweep)),
    "n_selected": len(selected), "n_scored": n_used, "nonfinite_ids": nonfinite,
    "order_seed": 0, "n_shared_with_p": len(set(selected) & set(p_ids)),
    "layout": LAYOUT, "roles": T_ROLES,
    "p2v_grid": p2v.round(6).tolist(), "v2p_grid": v2p.round(6).tolist(),
    "pc1_scores": cls["pc1_scores"].round(6).tolist(),
    "pc1_direction": cls["pc1_direction"], "pc1_mean": cls["pc1_mean"],
    "pc1_std": cls["pc1_std"], "sigma": cls["sigma"],
    "promoting": [list(h) for h in sorted(t_prom)],
    "suppressing": [list(h) for h in sorted(t_supp)],
    "records_file": "t_patching_records_qwen25vl3b.jsonl.gz",
}
envelope = result_envelope(
    "t_patching", summary,
    model="Qwen2.5-VL-3B-Instruct", checkpoint=config.checkpoint, seed=config.seed,
    prompt=f"visual={VISUAL_PROMPT!r}", n=n_used, repo_dir=REPO_DIR,
    decision="D71",
    quantisation="4-bit nf4, double quant",
    compute_dtype="bfloat16" if native_bf16(torch.cuda.get_device_capability(0)) else "float16",
    gpu=torch.cuda.get_device_name(0), image_tokens=n_img,
    method="last-token head patching (o_proj input slice), cached prefix; Lietzow PCA +/-2 sigma; "
           "source=incongruent note, target=neutral note",
)
print(write_result(WORK / "t_patching_qwen25vl3b.json", envelope))

# Per-object records gzipped: ~14 MB raw for ~280 objects, a few MB compressed.
# D70's bootstrap intervals need them, so they go in results/ too.
with gzip.open(WORK / "t_patching_records_qwen25vl3b.jsonl.gz", "wt") as f:
    for s in sweep:
        f.write(json.dumps(s) + "\n")
shutil.copy(WORK / "selection.jsonl", WORK / "t_patching_selection_qwen25vl3b.jsonl")
for name in ("t_patching_qwen25vl3b.json", "t_patching_records_qwen25vl3b.jsonl.gz",
             "t_patching_selection_qwen25vl3b.jsonl"):
    print(f"  {name:42s} {(WORK / name).stat().st_size / 1e6:6.2f} MB")
print("\nDownload these three into results/ and commit them.")